## CUAD clause detection: TF-IDF baseline
This notebook trains a multi-label classifier on Aditi's labeled contract chunks. It needs a ZIP containing `cuad_train_split.jsonl`, `cuad_val_split.jsonl`, and `cuad_test_split.jsonl`. A ZIP containing the original CUAD JSON files is not the processed dataset.
Run the cells in order. Use the validation split to assess and improve the baseline. Keep the test split untouched until the final evaluation.

### 1. Upload the processed splits
In Aditi's notebook, run through the final Splits saved. cell. Download the three JSONL outputs and place them together in a ZIP. Upload that ZIP when prompted below.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import pandas as pd

required = ["cuad_train_split.jsonl", "cuad_val_split.jsonl", "cuad_test_split.jsonl"]
print("Please upload the ZIP file containing the processed splits (cuad_train_split.jsonl, cuad_val_split.jsonl, cuad_test_split.jsonl) when prompted.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select one ZIP containing Aditi's three processed split files.")

zip_name = next(iter(uploaded))
data_dir = Path("/content/cuad_processed")
data_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_name) as archive:
    members = {Path(name).name: name for name in archive.namelist()}
    missing = [name for name in required if name not in members]
    if missing:
        raise ValueError(
            f"Missing processed files: {missing}. ZIP contains: {list(members)}. "
            "Run Aditi's notebook through 'Splits saved.' and ZIP its output JSONL files."
        )
    for name in required:
        (data_dir / name).write_bytes(archive.read(members[name]))

train_chunks = pd.read_json(data_dir / required[0], lines=True)
val_chunks = pd.read_json(data_dir / required[1], lines=True)
test_chunks = pd.read_json(data_dir / required[2], lines=True)

for name, frame in [("train", train_chunks), ("validation", val_chunks), ("test", test_chunks)]:
    needed = {"contract_id", "chunk_text", "labels"}
    assert needed.issubset(frame.columns), f"{name} missing columns: {needed - set(frame.columns)}"
    assert frame["labels"].map(lambda x: isinstance(x, list)).all(), f"{name} labels must be lists"
    print(name, "chunks:", len(frame), "contracts:", frame["contract_id"].nunique())

display(train_chunks[["contract_id", "chunk_text", "labels"]].head(3))

Please upload the ZIP file containing the processed splits (cuad_train_split.jsonl, cuad_val_split.jsonl, cuad_test_split.jsonl) when prompted.


Saving Archive.zip to Archive.zip
train chunks: 11500 contracts: 408
validation chunks: 1998 contracts: 51
test chunks: 1226 contracts: 51


,contract_id,chunk_text,labels
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,"[Document Name, Parties, Agreement Date]"
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"the State of Illinois, including but not li...","[Exclusivity, License Grant]"
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,distributors of the...,[]


### 2. Check that the data is suitable for evaluation
One contract must belong to only one split. A chunk may have zero, one, or several clause labels.

In [ ]:
train_ids = set(train_chunks["contract_id"])
val_ids = set(val_chunks["contract_id"])
test_ids = set(test_chunks["contract_id"])

assert not (train_ids & val_ids), "Train and validation share a contract"
assert not (train_ids & test_ids), "Train and test share a contract"
assert not (val_ids & test_ids), "Validation and test share a contract"

if "token_count" in train_chunks:
    assert max(df["token_count"].max() for df in [train_chunks, val_chunks, test_chunks]) <= 512

print("No contracts shared between splits.")
print("Training chunks by number of labels:")
print(train_chunks["labels"].map(len).value_counts().sort_index().head(12))
assert train_chunks["labels"].map(len).eq(0).any(), "No unlabeled training chunks: inspect processing"
assert train_chunks["labels"].map(len).gt(0).any(), "No positive training chunks: inspect processing"

No contracts shared between splits.
Training chunks by number of labels:
labels
0    7112
1    2575
2     980
3     450
4     268
5      71
6      29
7       8
8       7
Name: count, dtype: int64


### 3. Choose clause categories with training examples
CUAD uses names such as Cap On Liability and Uncapped Liability. The notebook prints unavailable or rare categories rather than inventing labels for them. These are clause-presence labels, not Low/Medium/High risk ratings.

In [ ]:
from collections import Counter
import pandas as pd

# Count labels in the correctly processed chunk datasets
train_counts = Counter(
    label
    for labels in train_chunks["labels"]
    for label in labels
)

val_counts = Counter(
    label
    for labels in val_chunks["labels"]
    for label in labels
)

# Model every clause category represented in training data
active_categories = sorted(train_counts.keys())

coverage = pd.DataFrame({
    "category": active_categories,
    "train_positive_chunks": [train_counts[c] for c in active_categories],
    "val_positive_chunks": [val_counts[c] for c in active_categories],
})

print("Number of categories:", len(active_categories))
display(coverage)


Number of categories: 41


,category,train_positive_chunks,val_positive_chunks
0,Affiliate License-Licensee,64,18
1,Affiliate License-Licensor,39,3
2,Agreement Date,386,55
3,Anti-Assignment,402,67
4,Audit Rights,314,71
5,Cap On Liability,407,57
6,Change Of Control,148,29
7,Competitive Restriction Exception,80,10
8,Covenant Not To Sue,113,17
9,Document Name,424,57


### 4. Train TF-IDF + logistic regression
The vectorizer learns vocabulary from training chunks only. The classifier learns a separate yes/no decision for each clause category.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer

label_encoder = MultiLabelBinarizer(classes=active_categories)
label_encoder.fit([[]])
selected = set(active_categories)
y_train = label_encoder.transform([[c for c in labels if c in selected] for labels in train_chunks["labels"]])
y_val = label_encoder.transform([[c for c in labels if c in selected] for labels in val_chunks["labels"]])

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True),
    OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000))
)
model.fit(train_chunks["chunk_text"].fillna(""), y_train)
print("Model trained on", len(train_chunks), "chunks.")

Model trained on 11500 chunks.


### 5. Evaluate on contracts the model has not seen
support is the number of positive validation chunks. Watch recall for important clauses, and interpret scores for categories with few examples cautiously.

In [ ]:
from sklearn.metrics import classification_report

predictions = model.predict(val_chunks["chunk_text"].fillna(""))
report = classification_report(
    y_val, predictions, target_names=label_encoder.classes_, zero_division=0,
    output_dict=True
)
results = pd.DataFrame(report).T
display(results.round(3))

per_category = results.loc[list(label_encoder.classes_), ["precision", "recall", "f1-score", "support"]]
per_category.to_csv("cuad_tfidf_validation_metrics.csv")
print("Saved per-category validation results: cuad_tfidf_validation_metrics.csv")

,precision,recall,f1-score,support
Affiliate License-Licensee,0.393,0.611,0.478,18.0
Affiliate License-Licensor,0.067,0.333,0.111,3.0
Agreement Date,0.490,0.855,0.623,55.0
Anti-Assignment,0.602,0.746,0.667,67.0
Audit Rights,0.667,0.789,0.723,71.0
Cap On Liability,0.518,0.772,0.620,57.0
Change Of Control,0.297,0.655,0.409,29.0
Competitive Restriction Exception,0.250,0.300,0.273,10.0
Covenant Not To Sue,0.464,0.765,0.578,17.0
Document Name,0.443,0.825,0.577,57.0


Saved per-category validation results: cuad_tfidf_validation_metrics.csv


### 6. Review a few errors
Read the clause text for a few false positives and missed clauses. This makes the baseline useful for deciding what to improve.

In [ ]:
import numpy as np

category = active_categories[0]  # Change this to any printed active category
idx = list(label_encoder.classes_).index(category)
false_negatives = np.flatnonzero((y_val[:, idx] == 1) & (predictions[:, idx] == 0))
false_positives = np.flatnonzero((y_val[:, idx] == 0) & (predictions[:, idx] == 1))

print("Category:", category)
print("Missed positive chunks:", len(false_negatives))
print("Incorrectly flagged chunks:", len(false_positives))
for name, positions in [("Missed clause", false_negatives), ("Incorrect flag", false_positives)]:
    for pos in positions[:2]:
        row = val_chunks.iloc[pos]
        print("\n ", name, "| contract:", row["contract_id"])
        print("True labels:", row["labels"])
        print("Text:", row["chunk_text"][:500])

Category: Affiliate License-Licensee
Missed positive chunks: 7
Incorrectly flagged chunks: 17

  Missed clause | contract: AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_Maintenance Agreement
True labels: ['License Grant', 'Affiliate License-Licensee', 'Irrevocable Or Perpetual License']
Text: or any statement of work, exhibit, schedule, order or other document under, subordinate to, or referencing this Agreement) for which AT&T has been or will be charged Structured Payments, Maintenance Fees or any monies in connection with a Change Order. Paid-For Development shall always include (without limitation) any modifications, alterations or updates that fall within the definition of Paid-For Development but shall not

Proprietary and Confidential This Agreement and information contained t

  Missed clause | contract: CardlyticsInc_20180112_S-1_EX-10.16_11002987_EX-10.16_Maintenance Agreement1
True labels: ['License Grant', 'Affiliate License-Licensee', 'Irrevocable Or Perpetual

### Reviewing False Positives and False Negatives Across All Active Categories

### What to do after this runs
1. Confirm the split counts, leakage check, and per-category metrics appear without errors.
2. Report validation precision, recall, F1, and support for each modeled clause category. Explain any categories that were unavailable or rare.
3. Review false positives and false negatives, then make one justified improvement if needed.
4. Evaluate on `test_chunks` only after model and thresholds are fixed.
5. Save this notebook and its metrics file to the team repository.

In [ ]:
# ============================================================
# FALSE POSITIVE / FALSE NEGATIVE ANALYSIS
# ACROSS ALL CLAUSE CATEGORIES
# ============================================================

import numpy as np
import pandas as pd
import re
from collections import Counter


# ------------------------------------------------------------
# 1. Make sure arrays are NumPy arrays
# ------------------------------------------------------------

y_val_array = np.asarray(y_val)
predictions_array = np.asarray(predictions)

print("=== DATA CHECK ===")
print("y_val shape:", y_val_array.shape)
print("predictions shape:", predictions_array.shape)
print("Number of categories:", len(label_encoder.classes_))
print("Total actual positives:", y_val_array.sum())
print("Total predicted positives:", predictions_array.sum())
print("Total prediction errors:",
      (y_val_array != predictions_array).sum())


# ------------------------------------------------------------
# 2. Count training examples for each clause category
# ------------------------------------------------------------

train_label_counts = Counter()

for labels in train_chunks["labels"]:
    for label in labels:
        train_label_counts[label] += 1


# ------------------------------------------------------------
# 3. Function for finding common words in error examples
# ------------------------------------------------------------

STOPWORDS = {
    "the", "and", "or", "of", "to", "a", "in", "for",
    "that", "this", "is", "be", "by", "with", "as",
    "on", "any", "an", "such", "shall", "may", "from",
    "its", "at", "if", "it", "are", "not", "which",
    "each", "all", "have", "has", "was", "were", "will",
    "their", "they", "there", "here", "under", "into"
}


def get_common_words(texts, n=10):

    words = []

    for text in texts:

        if pd.isna(text):
            continue

        tokens = re.findall(
            r"\b[a-zA-Z]{3,}\b",
            str(text).lower()
        )

        words.extend(
            word for word in tokens
            if word not in STOPWORDS
        )

    counts = Counter(words)

    return [
        word
        for word, count in counts.most_common(n)
    ]


# ------------------------------------------------------------
# 4. Analyze every clause category
# ------------------------------------------------------------

error_summary = []
error_examples = []

for idx, category in enumerate(label_encoder.classes_):

    actual = y_val_array[:, idx]
    predicted = predictions_array[:, idx]

    # Find TP, FP, FN
    tp_indices = np.flatnonzero(
        (actual == 1) & (predicted == 1)
    )

    fp_indices = np.flatnonzero(
        (actual == 0) & (predicted == 1)
    )

    fn_indices = np.flatnonzero(
        (actual == 1) & (predicted == 0)
    )

    # Number of examples
    train_count = train_label_counts.get(category, 0)
    val_positive_count = int(actual.sum())

    # Calculate metrics directly
    TP = len(tp_indices)
    FP = len(fp_indices)
    FN = len(fn_indices)

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else 0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    # --------------------------------------------------------
    # Get error text
    # --------------------------------------------------------

    fn_texts = [
        str(val_chunks.iloc[i]["chunk_text"])
        for i in fn_indices
    ]

    fp_texts = [
        str(val_chunks.iloc[i]["chunk_text"])
        for i in fp_indices
    ]

    common_fn_words = get_common_words(fn_texts)
    common_fp_words = get_common_words(fp_texts)

    # --------------------------------------------------------
    # Diagnose likely failure patterns
    # --------------------------------------------------------

    patterns = []

    if train_count < 10:
        patterns.append("Very limited training examples")

    elif train_count < 25:
        patterns.append("Limited training examples")

    if FN > FP and FN > 0:
        patterns.append(
            "Mostly false negatives / model misses clauses"
        )

    elif FP > FN and FP > 0:
        patterns.append(
            "Mostly false positives / similar language causes confusion"
        )

    elif FP > 0 and FN > 0:
        patterns.append(
            "Mixed false positives and false negatives"
        )

    if val_positive_count > 0 and recall < 0.50:
        patterns.append("Low recall")

    if (TP + FP) > 0 and precision < 0.50:
        patterns.append("Low precision")

    if val_positive_count > 0 and f1 < 0.50:
        patterns.append(
            "Weak overall performance"
        )

    if FP == 0 and FN == 0:
        patterns.append(
            "No validation errors for this category"
        )

    # --------------------------------------------------------
    # Add category summary
    # --------------------------------------------------------

    error_summary.append({

        "category": category,

        "train_examples": train_count,
        "validation_positives": val_positive_count,

        "true_positives": TP,
        "false_positives": FP,
        "false_negatives": FN,

        "precision": precision,
        "recall": recall,
        "f1": f1,

        "common_FN_words":
            ", ".join(common_fn_words),

        "common_FP_words":
            ", ".join(common_fp_words),

        "failure_patterns":
            "; ".join(patterns)
    })


    # --------------------------------------------------------
    # Store up to 5 FP and 5 FN examples per category
    # --------------------------------------------------------

    for pos in fn_indices[:5]:

        row = val_chunks.iloc[pos]

        error_examples.append({

            "category": category,
            "error_type": "False Negative",
            "contract_id": row.get(
                "contract_id",
                "Unknown"
            ),
            "true_labels": row["labels"],
            "text": row["chunk_text"]
        })


    for pos in fp_indices[:5]:

        row = val_chunks.iloc[pos]

        error_examples.append({

            "category": category,
            "error_type": "False Positive",
            "contract_id": row.get(
                "contract_id",
                "Unknown"
            ),
            "true_labels": row["labels"],
            "text": row["chunk_text"]
        })


# ------------------------------------------------------------
# 5. Build DataFrames
# ------------------------------------------------------------

error_summary_df = pd.DataFrame(
    error_summary,
    columns=[
        "category",
        "train_examples",
        "validation_positives",
        "true_positives",
        "false_positives",
        "false_negatives",
        "precision",
        "recall",
        "f1",
        "common_FN_words",
        "common_FP_words",
        "failure_patterns"
    ]
)


# IMPORTANT:
# Explicit columns prevent the KeyError you encountered
# when there happen to be zero error examples.

error_examples_df = pd.DataFrame(
    error_examples,
    columns=[
        "category",
        "error_type",
        "contract_id",
        "true_labels",
        "text"
    ]
)


# ------------------------------------------------------------
# 6. Sort worst-performing categories first
# ------------------------------------------------------------

error_summary_df = error_summary_df.sort_values(
    by=[
        "f1",
        "train_examples"
    ],
    ascending=[
        True,
        True
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Display full category analysis
# ------------------------------------------------------------

print("\n=== PER-CATEGORY FAILURE ANALYSIS ===")

display(
    error_summary_df[
        [
            "category",
            "train_examples",
            "validation_positives",
            "false_negatives",
            "false_positives",
            "precision",
            "recall",
            "f1",
            "failure_patterns"
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 8. Show underperforming categories
# ------------------------------------------------------------

underperforming = error_summary_df[
    (error_summary_df["f1"] < 0.60)
    | (error_summary_df["train_examples"] < 25)
    | (error_summary_df["false_negatives"] > 0)
    | (error_summary_df["false_positives"] > 0)
]


print("\n=== CATEGORIES NEEDING REVIEW ===")

if underperforming.empty:

    print("No categories met the underperformance criteria.")

else:

    display(
        underperforming[
            [
                "category",
                "train_examples",
                "f1",
                "false_negatives",
                "false_positives",
                "common_FN_words",
                "common_FP_words",
                "failure_patterns"
            ]
        ].round(3)
    )


# ------------------------------------------------------------
# 9. Display actual FP/FN contract examples
# ------------------------------------------------------------

print("\n=== FALSE POSITIVE / FALSE NEGATIVE EXAMPLES ===")

if error_examples_df.empty:

    print(
        "No false positive or false negative "
        "examples were found."
    )

else:

    display(
        error_examples_df[
            [
                "category",
                "error_type",
                "contract_id",
                "true_labels",
                "text"
            ]
        ]
    )


# ------------------------------------------------------------
# 10. Print concise written failure summary
# ------------------------------------------------------------

print("\n=== FAILURE PATTERN SUMMARY ===")

for _, row in error_summary_df.iterrows():

    if (
        row["false_positives"] > 0
        or row["false_negatives"] > 0
        or row["train_examples"] < 25
    ):

        print(f"\n{row['category']}")
        print(
            f"  Training examples: "
            f"{row['train_examples']}"
        )
        print(
            f"  FP: {row['false_positives']} | "
            f"FN: {row['false_negatives']}"
        )
        print(
            f"  Precision: {row['precision']:.3f} | "
            f"Recall: {row['recall']:.3f} | "
            f"F1: {row['f1']:.3f}"
        )
        print(
            f"  Pattern: "
            f"{row['failure_patterns']}"
        )


# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------

error_summary_df.to_csv(
    "cuad_error_analysis_by_category.csv",
    index=False
)

error_examples_df.to_csv(
    "cuad_error_examples.csv",
    index=False
)

print("\n=== SAVED ===")
print("cuad_error_analysis_by_category.csv")
print("cuad_error_examples.csv")

=== DATA CHECK ===
y_val shape: (1998, 41)
predictions shape: (1998, 41)
Number of categories: 41
Total actual positives: 1173
Total predicted positives: 1857
Total prediction errors: 1268

=== PER-CATEGORY FAILURE ANALYSIS ===


,category,train_examples,validation_positives,false_negatives,false_positives,precision,recall,f1,failure_patterns
0,Price Restrictions,20,3,3,0,0.000,0.000,0.000,Limited training examples; Mostly false negati...
1,Unlimited/All-You-Can-Eat-License,22,2,2,0,0.000,0.000,0.000,Limited training examples; Mostly false negati...
2,Most Favored Nation,25,6,6,0,0.000,0.000,0.000,Mostly false negatives / model misses clauses;...
3,Third Party Beneficiary,33,4,4,1,0.000,0.000,0.000,Mostly false negatives / model misses clauses;...
4,Affiliate License-Licensor,39,3,2,14,0.067,0.333,0.111,Mostly false positives / similar language caus...
5,No-Solicit Of Customers,33,8,7,0,1.000,0.125,0.222,Mostly false negatives / model misses clauses;...
6,Joint Ip Ownership,63,6,3,14,0.176,0.500,0.261,Mostly false positives / similar language caus...
7,Competitive Restriction Exception,80,10,7,9,0.250,0.300,0.273,Mostly false positives / similar language caus...
8,Non-Disparagement,38,8,6,4,0.333,0.250,0.286,Mostly false negatives / model misses clauses;...
9,Liquidated Damages,84,4,2,7,0.222,0.500,0.308,Mostly false positives / similar language caus...



=== CATEGORIES NEEDING REVIEW ===


,category,train_examples,f1,false_negatives,false_positives,common_FN_words,common_FP_words,failure_patterns
0,Price Restrictions,20,0.000,3,0,"pfhof, media, village, company, year, agreemen...",,Limited training examples; Mostly false negati...
1,Unlimited/All-You-Can-Eat-License,22,0.000,2,0,"hsns, piphany, licensed, program, period, appl...",,Limited training examples; Mostly false negati...
2,Most Favored Nation,25,0.000,6,0,"party, agreement, licensor, product, licensed,...",,Mostly false negatives / model misses clauses;...
3,Third Party Beneficiary,33,0.000,4,1,"agreement, party, third, parties, other, vendo...","agreement, provisions, party, rights, laws, co...",Mostly false negatives / model misses clauses;...
4,Affiliate License-Licensor,39,0.111,2,14,"party, honeywell, spinco, affiliates, section,...","nuance, spinco, license, use, agreement, roche...",Mostly false positives / similar language caus...
5,No-Solicit Of Customers,33,0.222,7,0,"agreement, airsopure, other, products, agent, ...",,Mostly false negatives / model misses clauses;...
6,Joint Ip Ownership,63,0.261,3,14,"work, product, lea, iger, ceres, intellectual,...","rights, property, intellectual, iger, party, c...",Mostly false positives / similar language caus...
7,Competitive Restriction Exception,80,0.273,7,9,"agreement, iger, parties, ceres, party, opport...","airsopure, agreement, products, you, other, co...",Mostly false positives / similar language caus...
8,Non-Disparagement,38,0.286,6,4,"agreement, company, professional, party, other...","agreement, airsopure, you, company, other, sec...",Mostly false negatives / model misses clauses;...
9,Liquidated Damages,84,0.308,2,7,"agreement, termination, consolidated, artists,...","company, year, section, agreement, during, fee...",Mostly false positives / similar language caus...



=== FALSE POSITIVE / FALSE NEGATIVE EXAMPLES ===


,category,error_type,contract_id,true_labels,text
0,Affiliate License-Licensee,False Negative,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,"[License Grant, Affiliate License-Licensee, Ir...","or any statement of work, exhibit, schedule, o..."
1,Affiliate License-Licensee,False Negative,CardlyticsInc_20180112_S-1_EX-10.16_11002987_E...,"[License Grant, Affiliate License-Licensee, Ir...","it, to compile it into Object Code and to prep..."
2,Affiliate License-Licensee,False Negative,CardlyticsInc_20180112_S-1_EX-10.16_11002987_E...,"[License Grant, Non-Transferable License, Affi...","Agreement, produce derivative works from and ..."
3,Affiliate License-Licensee,False Negative,ReynoldsConsumerProductsInc_20200121_S-1A_EX-1...,"[License Grant, Affiliate License-Licensor, Af...","as applicable, shall remain the property of su..."
4,Affiliate License-Licensee,False Negative,"PHREESIA,INC_05_28_2019-EX-10.18-STRATEGIC ALL...","[License Grant, Non-Transferable License, Affi...",performing validation and verification with re...
...,...,...,...,...,...
332,Warranty Duration,False Positive,"OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCIN...",[Cap On Liability],services.\n\n4.2 EXCLUSIVE REMEDIES. For any b...
333,Warranty Duration,False Positive,NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-D...,[],balances owed by NTC to ALFA AESAR. The partie...
334,Warranty Duration,False Positive,NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-D...,[Cap On Liability],defective or that fails to comply with the pur...
335,Warranty Duration,False Positive,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,[],in this Agreement (other than the specific war...



=== FAILURE PATTERN SUMMARY ===

Price Restrictions
  Training examples: 20
  FP: 0 | FN: 3
  Precision: 0.000 | Recall: 0.000 | F1: 0.000
  Pattern: Limited training examples; Mostly false negatives / model misses clauses; Low recall; Weak overall performance

Unlimited/All-You-Can-Eat-License
  Training examples: 22
  FP: 0 | FN: 2
  Precision: 0.000 | Recall: 0.000 | F1: 0.000
  Pattern: Limited training examples; Mostly false negatives / model misses clauses; Low recall; Weak overall performance

Most Favored Nation
  Training examples: 25
  FP: 0 | FN: 6
  Precision: 0.000 | Recall: 0.000 | F1: 0.000
  Pattern: Mostly false negatives / model misses clauses; Low recall; Weak overall performance

Third Party Beneficiary
  Training examples: 33
  FP: 1 | FN: 4
  Precision: 0.000 | Recall: 0.000 | F1: 0.000
  Pattern: Mostly false negatives / model misses clauses; Low recall; Low precision; Weak overall performance

Affiliate License-Licensor
  Training examples: 39
  FP: 14 | FN: 2
